# FLUX.2 Klein 4B — Garment T-pose Conversion + Gradio API

Auto-generated notebook for recipe: **flux2-tpose**


# FLUX.2 Klein 4B — Garment T-pose Conversion

단일 의류 사진을 **T-pose 정면 이미지**로 변환합니다.

- **Model**: `black-forest-labs/FLUX.2-klein-4B` (4B params, 4-step distilled)
- **Input**: 의류 사진 (평면촬영, 행거샷, 착용샷)
- **Output**: T-pose 의류 이미지 (흰배경, 1024x1024)
- **API**: Gradio 엔드포인트로 외부 호출 가능

---


## A) GPU Check


In [ ]:
#@title A) GPU & VRAM Check { run: "auto" }
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU not available! Change runtime: Runtime > Change runtime type > GPU")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")

if vram_gb < 12:
    print("WARNING: VRAM < 12GB. cpu_offload will be used but may be slow.")
else:
    print("OK: Sufficient VRAM for FLUX.2 Klein 4B")


## B) Install Dependencies


In [ ]:
#@title B) Install Dependencies { run: "auto" }
import subprocess, sys

# --- 1. Probe Flux2KleinPipeline ---
need_diffusers_upgrade = False
try:
    from diffusers import Flux2KleinPipeline
    import diffusers
    print(f"diffusers {diffusers.__version__} — Flux2KleinPipeline OK")
except ImportError:
    print("Flux2KleinPipeline not found in current diffusers. Upgrading to git HEAD...")
    need_diffusers_upgrade = True

if need_diffusers_upgrade:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/huggingface/diffusers.git"
    ])
    # Verify after upgrade
    from diffusers import Flux2KleinPipeline
    import diffusers
    print(f"diffusers {diffusers.__version__} (git HEAD) — Flux2KleinPipeline OK")
    print(">>> Colab kernel restart may be needed. Re-run this cell if import fails.")

# --- 2. rembg + gradio ---
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "rembg[gpu]", "gradio>=5.0", "accelerate>=1.12.0", "sentencepiece"
])

# --- 3. Verify ---
import torch, transformers, accelerate
print(f"\ntorch={torch.__version__}, CUDA={torch.version.cuda}")
print(f"transformers={transformers.__version__}")
print(f"accelerate={accelerate.__version__}")
print(f"CUDA available={torch.cuda.is_available()}")
print("\nAll dependencies installed.")


## C) HuggingFace Authentication

FLUX.2 Klein 4B는 Apache 2.0이지만, 다운로드를 위해 HF 토큰이 필요할 수 있습니다.


In [ ]:
#@title C) HuggingFace Login { run: "auto" }
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get("HF_TOKEN")
    login(token=hf_token)
    print("Logged in via Colab secret 'HF_TOKEN'")
except Exception:
    print("HF_TOKEN not found in Colab secrets.")
    print("If download fails, add your token: Colab sidebar > Secrets > HF_TOKEN")
    print("Or run: huggingface_hub.login()")


## D) Load FLUX.2 Klein 4B


In [ ]:
#@title D) Load Model { run: "auto" }
import torch
from diffusers import Flux2KleinPipeline

MODEL_ID = "black-forest-labs/FLUX.2-klein-4B"
DTYPE = torch.bfloat16

print(f"Loading {MODEL_ID} ...")
pipe = Flux2KleinPipeline.from_pretrained(MODEL_ID, torch_dtype=DTYPE)
pipe.enable_model_cpu_offload()

vram_after = torch.cuda.memory_allocated() / 1e9
print(f"Model loaded. VRAM used: {vram_after:.2f} GB (cpu_offload enabled)")
print("Ready for inference.")


## E) Preprocessing — Background Removal


In [ ]:
#@title E) Background Removal + Centering Functions
from rembg import remove
from PIL import Image
import numpy as np
import io

def remove_background(img: Image.Image) -> Image.Image:
    """Remove background using rembg, return RGBA image."""
    img_bytes = io.BytesIO()
    img.save(img_bytes, format="PNG")
    result_bytes = remove(img_bytes.getvalue())
    return Image.open(io.BytesIO(result_bytes)).convert("RGBA")

def center_on_white(rgba_img: Image.Image, target_size: int = 1024, padding: float = 0.05) -> Image.Image:
    """Center the garment on a white background with padding."""
    # Find bounding box of non-transparent pixels
    alpha = np.array(rgba_img)[:, :, 3]
    rows = np.any(alpha > 10, axis=1)
    cols = np.any(alpha > 10, axis=0)

    if not rows.any() or not cols.any():
        # Fallback: return white image
        return Image.new("RGB", (target_size, target_size), (255, 255, 255))

    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]

    # Crop to content
    cropped = rgba_img.crop((cmin, rmin, cmax + 1, rmax + 1))
    cw, ch = cropped.size

    # Calculate target area with padding
    pad_px = int(target_size * padding)
    available = target_size - 2 * pad_px
    scale = min(available / cw, available / ch)
    new_w = int(cw * scale)
    new_h = int(ch * scale)

    cropped_resized = cropped.resize((new_w, new_h), Image.LANCZOS)

    # Paste onto white background
    result = Image.new("RGB", (target_size, target_size), (255, 255, 255))
    offset_x = (target_size - new_w) // 2
    offset_y = (target_size - new_h) // 2
    result.paste(cropped_resized, (offset_x, offset_y), cropped_resized)

    return result

def preprocess_garment(img: Image.Image, target_size: int = 1024) -> Image.Image:
    """Full preprocessing: remove bg + center on white."""
    rgba = remove_background(img)
    return center_on_white(rgba, target_size)

print("Preprocessing functions loaded.")


## F) T-pose Prompt Builder

프롬프트 엔지니어링 전략:
- **Subject + Action + Style + Context** 구조
- 100단어 이내 (Klein 최적)
- T-pose/flat-lay를 맨 앞에 (decoder-only causal attention)
- No negative prompts (FLUX.2 미지원)


In [ ]:
#@title F) Prompt Builder

GARMENT_TYPES = {
    "t-shirt": "short-sleeve t-shirt",
    "hoodie": "hooded sweatshirt",
    "jacket": "zip-up jacket",
    "shirt": "button-down collared shirt",
    "sweater": "knit pullover sweater",
    "coat": "long coat",
    "dress": "dress",
    "polo": "polo shirt",
    "vest": "vest",
    "custom": "",
}

def build_tpose_prompt(garment_type: str = "t-shirt", custom_desc: str = "") -> str:
    """Build the T-pose conversion prompt."""
    if garment_type == "custom" and custom_desc:
        garment_desc = custom_desc
    else:
        garment_desc = GARMENT_TYPES.get(garment_type, "garment")

    prompt = (
        f"Flat-lay product photograph of a {garment_desc} laid flat in T-pose position. "
        f"Both sleeves are spread out horizontally to the sides, "
        f"front of the garment facing directly toward the camera. "
        f"The garment preserves the exact same fabric texture, colors, patterns, and any logos or prints from the reference. "
        f"Centered on a pure white background. "
        f"Studio lighting with soft diffused shadows, sharp focus, high detail. "
        f"Professional e-commerce product photography, 85mm lens, f/5.6."
    )
    return prompt

# Preview
sample_prompt = build_tpose_prompt("t-shirt")
word_count = len(sample_prompt.split())
print(f"Sample prompt ({word_count} words):")
print(sample_prompt)


## G) Single Inference Test

의류 이미지를 업로드하고 T-pose 변환을 테스트합니다.


In [ ]:
#@title G) Upload & Convert to T-pose
import torch
from PIL import Image
from google.colab import files
import time

#@markdown ### Settings
GARMENT_TYPE = "t-shirt"  #@param ["t-shirt", "hoodie", "jacket", "shirt", "sweater", "coat", "dress", "polo", "vest", "custom"]
CUSTOM_DESC = ""  #@param {type:"string"}
STRENGTH = 0.55  #@param {type:"slider", min:0.3, max:0.7, step:0.05}
SEED = 42  #@param {type:"integer"}
TARGET_SIZE = 1024  #@param [768, 1024] {type:"raw"}

# --- Upload ---
print("Upload a garment image:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
garment_img = Image.open(filename).convert("RGB")
print(f"Uploaded: {filename} ({garment_img.size})")

# --- Preprocess ---
print("Removing background...")
preprocessed = preprocess_garment(garment_img, TARGET_SIZE)
display(preprocessed)
print(f"Preprocessed: {preprocessed.size}")

# --- Build prompt ---
prompt = build_tpose_prompt(GARMENT_TYPE, CUSTOM_DESC)
print(f"\nPrompt: {prompt}")

# --- Inference ---
print(f"\nGenerating T-pose (strength={STRENGTH}, steps=4, seed={SEED})...")
generator = torch.Generator(device="cpu").manual_seed(SEED)

t0 = time.time()
result = pipe(
    prompt=prompt,
    image=preprocessed,
    strength=STRENGTH,
    height=TARGET_SIZE,
    width=TARGET_SIZE,
    guidance_scale=1.0,
    num_inference_steps=4,
    generator=generator,
).images[0]
elapsed = time.time() - t0

print(f"Done in {elapsed:.1f}s")
display(result)

# --- Save ---
out_path = f"tpose_{filename}"
result.save(out_path)
print(f"Saved: {out_path}")


## H) Gradio App + API Endpoint

`share=True`로 공개 URL이 생성됩니다.
Gradio Client로 API 호출 가능:

```python
from gradio_client import Client
client = Client("https://xxxxx.gradio.live")
result = client.predict(
    image="garment.jpg",
    garment_type="t-shirt",
    strength=0.55,
    seed=42,
    api_name="/convert"
)
```


In [ ]:
#@title H) Launch Gradio App + API { run: "auto" }
import gradio as gr
import torch
from PIL import Image
import time
import numpy as np

def convert_to_tpose(
    image: Image.Image,
    garment_type: str = "t-shirt",
    custom_desc: str = "",
    strength: float = 0.55,
    seed: int = 42,
    target_size: int = 1024,
) -> tuple[Image.Image, Image.Image, str]:
    """Convert garment image to T-pose. Returns (preprocessed, result, info)."""
    if image is None:
        raise gr.Error("Please upload a garment image.")

    image = Image.fromarray(image) if isinstance(image, np.ndarray) else image
    image = image.convert("RGB")

    # Preprocess
    preprocessed = preprocess_garment(image, target_size)

    # Build prompt
    prompt = build_tpose_prompt(garment_type, custom_desc)

    # Inference
    generator = torch.Generator(device="cpu").manual_seed(seed)
    t0 = time.time()
    result = pipe(
        prompt=prompt,
        image=preprocessed,
        strength=strength,
        height=target_size,
        width=target_size,
        guidance_scale=1.0,
        num_inference_steps=4,
        generator=generator,
    ).images[0]
    elapsed = time.time() - t0

    info = f"Time: {elapsed:.1f}s | Strength: {strength} | Seed: {seed} | Type: {garment_type}"
    return preprocessed, result, info

# --- Build Gradio UI ---
with gr.Blocks(title="Garment T-pose Converter") as demo:
    gr.Markdown("# Garment T-pose Converter\nUpload a garment photo to convert it to T-pose flat-lay format.")

    with gr.Row():
        with gr.Column():
            input_image = gr.Image(label="Input Garment", type="pil")
            garment_type = gr.Dropdown(
                choices=list(GARMENT_TYPES.keys()),
                value="t-shirt",
                label="Garment Type",
            )
            custom_desc = gr.Textbox(label="Custom Description (if type=custom)", visible=True)
            strength = gr.Slider(0.3, 0.7, value=0.55, step=0.05, label="Strength (denoising)")
            seed = gr.Number(value=42, label="Seed", precision=0)
            target_size = gr.Radio([768, 1024], value=1024, label="Output Size")
            btn = gr.Button("Convert to T-pose", variant="primary")

        with gr.Column():
            preprocessed_output = gr.Image(label="Preprocessed (bg removed)")
            result_output = gr.Image(label="T-pose Result")
            info_output = gr.Textbox(label="Info")

    btn.click(
        fn=convert_to_tpose,
        inputs=[input_image, garment_type, custom_desc, strength, seed, target_size],
        outputs=[preprocessed_output, result_output, info_output],
        api_name="convert",
    )

# --- Launch ---
print("Launching Gradio app with public URL...")
demo.launch(share=True, quiet=False)
